# Ejercicio 2 – MEC (2048)

Este notebook sirve como guía y bitácora para documentar el **Ejercicio 2** del obligatorio de Inteligencia Artificial.
En esta sección abordaremos el entorno **2048**, implementaremos agentes basados en técnicas de búsqueda (Expectimax y Minimax con poda α‑β), definiremos funciones de evaluación, diseñaremos experimentos controlados y registraremos sus resultados.

La estructura sigue el mismo formato utilizado en el informe del Ejercicio 1: introducción, entendimiento del problema, técnicas, evaluación, experimentación, análisis de resultados y conclusiones.

## 1. Entendimiento del problema

**2048** es un juego de tablero de tamaño 4×4 en el que el jugador debe combinar fichas con valores potencia de 2 para alcanzar la ficha `2048`.
En cada turno ocurren dos fases:
1. El jugador elige una dirección (`UP=0`, `DOWN=1`, `LEFT=2`, `RIGHT=3`) y todas las fichas se desplazan en esa dirección, combinándose cuando dos fichas iguales chocan.
2. Se inserta aleatoriamente una ficha nueva de valor `2` (con probabilidad 0,9) o `4` (con probabilidad 0,1) en una de las celdas vacías.

El agente debe maximizar la probabilidad de alcanzar altas fichas (idealmente 2048) manteniendo el tablero jugable (evitando bloqueos).

**Convención de movimientos:** en este notebook seguimos la convención de `GameBoard.dirs` (0=UP, 1=DOWN, 2=LEFT, 3=RIGHT).

## 2. Técnicas implementadas

### 2.1 Expectimax
- Se modela el turno del jugador como un nodo **MAX** (se elige el movimiento con mayor valor esperado).
- La inserción de la ficha se modela como un nodo **CHANCE**: se considera cada celda vacía y se calcula un valor esperado ponderando 2 y 4 con probabilidades 0,9 y 0,1.
- Se expande el árbol hasta una profundidad dada (`depth`), evaluando los estados terminales con una función heurística.
- Para preservar la aleatoriedad del juego, las llamadas a `clone()` y `get_available_moves()` están envueltas en un context manager (`preserve_numpy_rng`).

### 2.2 Minimax con poda α‑β
- Se modela el turno del jugador como un nodo **MAX**.
- Se modela la inserción de ficha como un nodo **MIN** adversarial (elige la peor casilla y valor de ficha). Esto simula un oponente que intenta minimizar el valor del jugador.
- Se implementa la poda **α‑β** para cortar ramas cuando ya no pueden superar la mejor solución encontrada. También se incluye la opción de desactivar la poda (`use_pruning=False`) para comparar rendimientos.
- Al igual que en Expectimax, se usa `preserve_numpy_rng` para que la simulación no avance la secuencia aleatoria real.

## 3. Funciones de evaluación

Para evaluar un estado del tablero se combinan múltiples **heurísticas** (funciones de evaluación intermedias) ponderadas por pesos. Estas heurísticas están implementadas en `heuristics.py`.

Las principales heurísticas son:
- **count_empty**: número de celdas vacías (un tablero con espacio es más flexible).
- **monotonicity**: mide cuán ordenadas (monótonas) son las filas y columnas al considerar log₂ de las fichas.
- **smoothness**: penaliza grandes diferencias entre celdas adyacentes. Se calcula sobre log₂ y devuelve un valor negativo; multiplicado por un peso positivo actúa como recompensa.
- **max_tile_in_corner**: recompensa que la ficha máxima esté en una esquina.
- **merges_possible**: cuenta pares de fichas adyacentes iguales (potenciales merges).
- **positional_weight**: pondera una distribución serpenteada que coloca las fichas grandes en la esquina superior izquierda.

La combinación se realiza en la función `evaluate(board_or_grid, weights)` como suma de `peso_i * heurística_i`.
Se han definido múltiples **presets** de pesos para explorar distintas estrategias: `baseline`, `tuned`, `corner_heavy`, `smooth_heavy`, `snake`, etc. Estos presets están disponibles en `bench.py`.

## 4. Diseño de experimentos

Las pruebas recomendadas para documentar el ejercicio son:
- **Comparación de pesos (presets):** mantener fijo el agente (por ejemplo Expectimax con profundidad 3) y comparar los presets (`baseline`, `tuned`, `corner_heavy`, `smooth_heavy`, `snake`).
- **Comparación de profundidades:** para un preset determinado (por ejemplo `smooth_heavy`), comparar profundidades de búsqueda (`depth=2,3,4`) tanto para Expectimax como para Minimax.
- **Impacto de la poda α‑β:** correr Minimax con y sin poda para una combinación de preset y profundidad (ej. `depth=3`, preset `smooth_heavy`) y medir la diferencia de tiempo/resultado.
- **Agente aleatorio:** ejecutar varias partidas con `RandomAgent` para tener una línea base de rendimiento.

Para cada configuración (agente, profundidad, preset) se deben correr **varios episodios** (por ejemplo 30 o más) con una semilla fija (`--seed`) para reproducibilidad. El script `bench.py` permite seleccionar estas opciones y guarda los resultados por episodio en un archivo CSV.

Los parámetros clave son:
- `--agent`: `expectimax`, `minimax` o `random`.
- `--depth`: profundidad de búsqueda (solo para `expectimax` y `minimax`).
- `--preset` o `--weights`: selecciona el preset de pesos o define pesos personalizados.
- `--episodes`: número de partidas a correr.
- `--seed`: semilla de NumPy para reproducibilidad (opcional).
- `--no-pruning`: desactiva la poda para Minimax.

A continuación se muestran celdas de ejemplo para ejecutar corridas y guardar los CSV en la carpeta `results/`. Puedes ajustar los parámetros según tus necesidades.

### Crear directorio de resultados

Antes de ejecutar las corridas, crea un directorio para almacenar los archivos CSV con los resultados.
Desde la terminal (shell), ejecuta:

```bash
mkdir -p results
```
Esto creará la carpeta `results/` si no existe.

### Ejecutar experimentos desde la terminal

A continuación se listan los comandos que puedes ejecutar desde la terminal para correr las partidas y guardar los resultados. Ajusta los valores de `--episodes`, `--seed`, `--depth` y el preset según tus necesidades.

Para Expectimax con distintos presets y profundidades:
```bash
python bench.py --agent expectimax --depth 2 --preset smooth_heavy   --episodes 30 --seed 0 --output results/expectimax_smooth_d2_e30.csv
python bench.py --agent expectimax --depth 3 --preset smooth_heavy   --episodes 30 --seed 0 --output results/expectimax_smooth_d3_e30.csv
python bench.py --agent expectimax --depth 4 --preset smooth_heavy   --episodes 30 --seed 0 --output results/expectimax_smooth_d4_e30.csv

# Comparación de presets en depth=3
python bench.py --agent expectimax --depth 3 --preset baseline       --episodes 30 --seed 0 --output results/expectimax_baseline_d3_e30.csv
python bench.py --agent expectimax --depth 3 --preset tuned          --episodes 30 --seed 0 --output results/expectimax_tuned_d3_e30.csv
python bench.py --agent expectimax --depth 3 --preset corner_heavy   --episodes 30 --seed 0 --output results/expectimax_corner_heavy_d3_e30.csv
```

Para Minimax con y sin poda (α‑β):
```bash
python bench.py --agent minimax --depth 3 --preset smooth_heavy --episodes 20 --seed 0 --output results/minimax_smooth_d3_prune_on_e20.csv
python bench.py --agent minimax --depth 3 --preset smooth_heavy --episodes 20 --seed 0 --no-pruning --output results/minimax_smooth_d3_prune_off_e20.csv
```

Para la línea base aleatoria:
```bash
python bench.py --agent random --episodes 30 --seed 0 --output results/random_e30.csv
```

## 5. Análisis de resultados

Para analizar los resultados generados por las corridas anteriores, podemos leer los CSV desde el directorio `results/` y calcular métricas como:
- **Tasa de victoria** (`win`) – proporción de partidas en las que se alcanzó la ficha 2048.
- **Ficha máxima promedio** – promedio del valor máximo alcanzado en cada episodio.
- **Número de movimientos promedio** – duración promedio de las partidas.
- **Tiempo promedio por episodio** – duración en segundos.

A continuación se muestra un código de ejemplo para leer todos los CSV de `results/`, agrupar por agente/preset/profundidad y mostrar una tabla resumen. Después de ejecutar tu propia matriz de experimentos, reemplaza las rutas y ajusta según corresponda.

In [ ]:
import pandas as pd
from pathlib import Path

# Leer todos los CSV generados
def load_results(path_pattern='results/*.csv'):
    files = list(Path('.').glob(path_pattern))
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df['file'] = f.name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# Cargar y procesar
results_df = load_results()

if not results_df.empty:
    # Derivar columnas de configuración a partir del nombre de archivo (puedes personalizar)
    def parse_info(row):
        name = row['file']
        agent = None
        depth = None
        preset = None
        if 'expectimax' in name:
            agent = 'expectimax'
        elif 'minimax' in name:
            agent = 'minimax'
        elif 'random' in name:
            agent = 'random'
        # detectar profundidad en nombre
        for d in [2, 3, 4]:
            if f'd{d}' in name:
                depth = d
                break
        # detectar preset
        presets = ['baseline', 'tuned', 'corner_heavy', 'smooth_heavy', 'snake']
        for p in presets:
            if p in name:
                preset = p
                break
        return pd.Series({'agent': agent, 'depth': depth, 'preset': preset})

    cfg_cols = results_df.apply(parse_info, axis=1)
    results_df = pd.concat([results_df, cfg_cols], axis=1)

    # Calcular métricas agregadas
    summary = results_df.groupby(['agent', 'depth', 'preset']).agg(
        episodes=('win', 'count'),
        win_rate=('win', 'mean'),
        avg_max_tile=('max_tile', 'mean'),
        avg_moves=('moves', 'mean'),
        avg_duration_sec=('duration_sec', 'mean')
    ).reset_index()
    
    # Mostrar tabla
    from IPython.display import display
    display(summary)
else:
    print('No se encontraron resultados en la carpeta results/. Ejecuta las corridas primero.')


## 6. Conclusiones y discusión

Aquí se incluyen tus conclusiones después de analizar los resultados. Algunas ideas para discutir:
- ¿Qué combinación de agente, profundidad y preset se comportó mejor?
- ¿Cómo afecta la profundidad al rendimiento y al tiempo de cómputo?
- ¿Cuál es el impacto de usar poda α‑β en Minimax?
- Comparación del agente Expectimax contra Minimax y Random.

El análisis debe acompañarse de gráficos o tablas que evidencien tus observaciones.

**Aclaración sobre uso de IAG:** este notebook y sus scripts utilizan la herramienta generativa (ChatGPT) como asistente de programación y documentación. Todas las decisiones de diseño (heurísticas, pesos, número de episodios, etc.) deben ser justificadas en el informe y validadas ejecutando los experimentos.